In [1]:
import numpy as np
import pandas as pd
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image

# ----------------------------
# Device
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ----------------------------
# Paths
# ----------------------------
TEST_CSV = '/kaggle/input/csiro-biomass/test.csv'
TEST_IMG_DIR = '/kaggle/input/csiro-biomass/test'
MODEL_DIR = '/kaggle/input/csirow-b3-models-v2/'  # ← YOUR DATASET NAME

# ----------------------------
# Normalization stats (from your 85% train split in Version 2)
# These values are hardcoded from your training — they MUST match!
outer_mean = np.array([6.6497, 12.0445, 26.6247, 45.3181, 33.2744])  # [Clover, Dead, Green, Total, GDM]
outer_std = np.array([12.1178, 12.4020, 25.4012, 27.9840, 24.9358])
target_columns = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']

# ----------------------------
# Model definition (MUST match Version 2)
# ----------------------------
def create_model():
    # Use pre-trained weights — but since internet is OFF, it will load from cache IF previously downloaded.
    # However, we avoid download by using weights=None + manual head (safe for submission)
    model = models.efficientnet_b3(weights=None) 
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 5)
    )
    return model

# ----------------------------
# Test Dataset
# ----------------------------
class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        img = Image.open(os.path.join(self.img_dir, img_path)).convert('RGB')
        return self.transform(img), img_path

# ----------------------------
# Transforms
# ----------------------------
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ----------------------------
# Load test data
# ----------------------------
df_test = pd.read_csv(TEST_CSV)
df_test['image_path'] = df_test['image_path'].str.replace(r'^test/', '', regex=True)
df_unique = df_test[['image_path']].drop_duplicates().reset_index(drop=True)
print(f"Predicting on {len(df_unique)} images")

# ----------------------------
# Load ensemble models
# ----------------------------
model_paths = [
    f"{MODEL_DIR}best_model_fold_test_efficient_b3_0.pth",
    f"{MODEL_DIR}best_model_fold_test_efficient_b3_1.pth",
    f"{MODEL_DIR}best_model_fold_test_efficient_b3_2.pth"
]

trained_ensemble_models = []
for path in model_paths:
    model = create_model()
    # Load state dict directly (your models were saved as raw state dicts)
    state_dict = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(state_dict)
    model.to(device).eval()
    trained_ensemble_models.append(model)

# ----------------------------
# Inference function
# ----------------------------
def ensemble_predict(models_list, images, device):
    all_preds = []
    with torch.no_grad():
        for model in models_list:
            preds = model(images.to(device))
            all_preds.append(preds)
    return torch.stack(all_preds).mean(dim=0)

# ----------------------------
# Run inference
# ----------------------------
test_ds = TestDataset(df_unique, TEST_IMG_DIR, eval_transform)
test_dl = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

all_img_paths = []
all_preds = []

with torch.no_grad():
    for imgs, paths in test_dl:
        preds = ensemble_predict(trained_ensemble_models, imgs, device)
        all_preds.append(preds.cpu())
        all_img_paths.extend(paths)

preds = torch.cat(all_preds, dim=0).numpy()
# Denormalize to grams
preds_orig = preds * outer_std + outer_mean

# ----------------------------
# Create submission.csv
# ----------------------------
rows = []
for i, img_path in enumerate(all_img_paths):
    img_id = os.path.splitext(os.path.basename(img_path))[0]  # e.g., "ID1001187975"
    for j, target in enumerate(target_columns):
        sample_id = f"{img_id}__{target}"
        rows.append({"sample_id": sample_id, "target": float(preds_orig[i, j])})

submission = pd.DataFrame(rows)[['sample_id', 'target']]
submission.to_csv('submission.csv', index=False)

print(" Submission saved!")
print(submission.head())

Using device: cuda
Predicting on 1 images
 Submission saved!
                    sample_id     target
0  ID1001187975__Dry_Clover_g   0.868679
1    ID1001187975__Dry_Dead_g  20.082540
2   ID1001187975__Dry_Green_g  38.864819
3   ID1001187975__Dry_Total_g  58.407570
4         ID1001187975__GDM_g  37.733824
